# Ophidia-based workflow using ML4Fires

This workflow can be used to test the inference model for Wildfire Burned Areas Prediction on climate data. It allows to evaluate burned areas results across an ensemble of different climate models and scenarios.<br/>
The workflow includes a number of tasks for data preparation, ML model execution and data post-processing. In particular, given a set of input parameters (e.g. "time domain" can be set through the command line), the workflow:

- selects the variables from input NetCDF files,
- aggregates them accordingly (different operations can be applied to the various variables),
- regrids the variables to a common grid,
- executes a pre-trained ML model on such data,
- masks the results (seas and poles),
- aggregates the results on yearly basis, and
- computes the statistics from the ensemble of results on multiple CMIP6 models.

First of all, set the input parameters:

- time range
- list of scenarios
- list of models.

In [ ]:
import ipywidgets as widgets

scenario = widgets.Dropdown(
    options=[('SSP126', 'ssp126'), ('SSP245', 'ssp245'), ('SSP370', 'ssp370'),
             ('SSP585', 'ssp585')],
    value = 'ssp126',
    style={'description_width': '60px'}, 
    description='Scenario', disabled=False,
    layout=widgets.Layout(width='200px'))

climate_model = widgets.SelectMultiple(
    options=[('MPI-ESM1-2-HR', 'MPI-ESM1-2-HR'), ('CMCC-ESM2', 'CMCC-ESM2'), 
             ('NorESM2-MM', 'NorESM2-MM'), ('CESM2', 'CESM2')],
    value = ['CMCC-ESM2'],
    style={'description_width': '60px'}, 
    description='Model', disabled=False,
    layout=widgets.Layout(width='200px'))

year_range = widgets.IntRangeSlider(
    value=[2030, 2035],        # initial range
    min=2015,                 # min value
    max=2100,               # max value
    step=1,                # step size
    description='Year range:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='400px')
)

display(widgets.HBox([scenario, climate_model, year_range]))

And then, execute the workflow.

In [ ]:
from fires import fires

time_range = str(year_range.value[0]) + "-01-01_" + str(year_range.value[1]) + "-12-31"
scenarios = [scenario.value]
models = list(climate_model.value)
output_folder = fires(time_range = time_range, scenarios = scenarios, models = models)

Finally, plot the results from the ensemble analysis.

In [ ]:
import xarray as xr
import sys
sys.path.append('../')
from Fires._utilities.utils_inference import process_and_plot_data, load_input_data

searfire_ds_path = "../../ML4Fires_data/data_100km.zarr"
input_data = load_input_data(searfire_ds_path, '2019', '2020') # Required by internal setting for creating the land-sea mask

ds = xr.open_dataset(output_folder + "avg_" + scenario.value + ".nc", engine="netcdf4")
process_and_plot_data(
	data=ds.global_burned_areas,
	label='Predicted Burned Areas',
	lats=ds.lat.values,
	lons=ds.lon.values,
	model_name="Unet ++"
)